# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [ ]:
import os

from dotenv import load_dotenv  # loads variables from .env into os.environ
from IPython.display import Markdown, display  # render LLM output as markdown
from openai import OpenAI  # official OpenAI Python client

from scraper import fetch_website_content  # local helper, see scraper.py

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [5]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [ ]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [ ]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

Title: Azam Mustufa — Software Engineer

Azam Mustufa — Software Engineer
AZAM.DEV
Work
Experience
Connect
mail
System Operational · IBM, Bangalore
Software Engineer
at IBM
Software Engineer at IBM working on SAP SuccessFactors cloud integrations. I build backend systems with Java, Spring Boot, and Node.js, and focus on quality engineering, test automation, and distributed systems.
281+
LeetCode · Top 13%
5
Projects deployed
3×
Hackathon finalist
View Work
arrow_downward
Connect
Projects
Backend Systems
terminal
Live
Database Backup CLI Infrastructure
Production-grade automation CLI for async database backups — reduces manual deployment and migration time by 80%, handles 50GB migrations under 150MB memory with 99.9% success rate.
TypeScript
MongoDB
MySQL
Node.js
Linux
Bash
GitHub
arrow_outward
hub
Live
Project Management Engine
Scalable REST backend with 25+ endpoints and granular RBAC. API responses optimised to under 120ms via Redis caching. BullMQ async pipelines cut cache latency b

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [ ]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [ ]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [ ]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [18]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

**GPT:** # Summary of the Website

- Azam Mustufa is a Software Engineer based in Bangalore, currently working at IBM on SAP SuccessFactors cloud integrations.
- Focus areas: backend systems, Java and Spring Boot, Node.js, quality engineering, test automation, and distributed systems.
- Notable metrics: 30+ defects resolved during cloud integration testing; 15% improvement in release stability; 3× Hackathon finalist; 5 projects deployed; LeetCode with 281+ problems (Top 13%).
- Key projects showcased (backend and infrastructure): Production-grade Database Backup CLI, Project Management Engine, Distributed Image Processing Backend, HTTP Caching Proxy, and UPI Offline Mesh. Each project highlights scalable architectures and specific tech stacks.
- Technical stack includes: Java, JavaScript/TypeScript, Python, C++, SQL; Spring Boot, Spring Security, Node.js, Express, Bun, BullMQ, Mongoose; MongoDB, MySQL, Redis, RabbitMQ; Docker, Linux, CI/CD; testing with JUnit, Mockito, Selenium, Playwright, Cypress.
- Education: B.E. in Computer Science & Engineering from Visvesvaraya Technological University (CGPA 7.28/10); notable coursework in OS, networks, data structures, databases, software engineering; 3× Hackathon finalist.
- Availability: Accepting new requests; located in Bangalore, India; open to remote work.
- Contact and online presence: Email, resume, GitHub, LinkedIn, LeetCode profile.
- Site note: AZAM.DEV built with Nuxt 3.